# Project 1 Notebook Walkthrough: Calgary Spatial ETL Pipeline

This notebook is your guided learning companion for **Project 1**.
It explains what each step does, why it matters, and what to run.

Scope covered:
1. Project context and environment setup
2. PostGIS setup with Docker
3. Extract module
4. Transform module
5. QA quality gate
6. Transactional PostGIS Load
7. End-to-end runner and verification

## How to Use This Notebook

- Read each markdown section first to understand the purpose.
- Run code cells in order.
- If a command fails, read the explanation below that section before moving on.
- Keep your terminal open for Docker and conda commands when needed.

## Kernel Setup (One-Time)

If VS Code prompts you to install `ipykernel`, that is normal for a new environment.
`ipykernel` is not part of ETL logic, it is the bridge that lets Jupyter run code in your selected Python environment.

Run these in terminal once:
```bash
conda activate calgary-etl
conda install ipykernel -y
python -m ipykernel install --user --name calgary-etl --display-name "Python (calgary-etl)"
```

Then select notebook kernel: **Python (calgary-etl)**.

## Step 0: Confirm Project Context

**Purpose:** Verify that your notebook is running from the correct project folder and that key files exist.

Why this matters:
- Many pipeline errors are caused by wrong working directory.
- Early file checks save debugging time later.

In [4]:
# Path helps us work with files/folders in a cross-platform way.
from pathlib import Path
# os is used here to change the working directory.
import os
# sys lets us inspect the active Python executable (kernel).
import sys

# This function finds the project root by searching upward for key files/folders.
def find_project_root(start: Path) -> Path:
    # These are the markers that define this repository root.
    markers = ["environment.yml", "docker-compose.yml", "src"]
    # Resolve to an absolute, normalized path.
    current = start.resolve()
    # Check current directory first, then each parent directory.
    for candidate in [current, *current.parents]:
        # candidate / m builds a child path; all(...) requires every marker to exist.
        if all((candidate / m).exists() for m in markers):
            # First directory that matches all markers is our project root.
            return candidate
    # Fallback: if nothing matches, keep the original start directory.
    return start.resolve()

# Capture the starting working directory for reporting.
start_cwd = Path.cwd()
# Find where the repository root should be.
project_root = find_project_root(start_cwd)

# If notebook started in a subfolder (for example docs), switch to project root.
if project_root != start_cwd:
    os.chdir(project_root)
    print(f"Changed working directory: {start_cwd} -> {project_root}")
else:
    # If already correct, report that no change was needed.
    print(f"Working directory already at project root: {project_root}")

# Re-read cwd after possible directory change.
cwd = Path.cwd()
print(f"Current working directory: {cwd}")
# Show exactly which Python environment the notebook kernel is using.
print(f"Python executable (active kernel): {sys.executable}")

# These are files we expect to exist in a correctly initialized project.
expected = [
    "environment.yml",
    "docker-compose.yml",
    "src/config.py",
    "src/extract.py",
]

# Verify each expected path and print a simple status line.
for rel in expected:
    path = cwd / rel
    print(f"{'OK' if path.exists() else 'MISSING'} - {rel}")

Working directory already at project root: /home/kanon/calgary-spatial-etl
Current working directory: /home/kanon/calgary-spatial-etl
Python executable (active kernel): /home/kanon/miniconda3/envs/calgary-etl/bin/python
OK - environment.yml
OK - docker-compose.yml
OK - src/config.py
OK - src/extract.py


## Step 1: Environment Setup

**Purpose:** Create a reproducible Python environment with geospatial dependencies.

Why this matters:
- Geospatial libraries depend on native binaries (GDAL/PROJ).
- A pinned environment makes your project portable and reviewable.

Run these in terminal (recommended):
```bash
conda env create -f environment.yml
conda activate calgary-etl
```

Then run the next cell to validate imports in the active kernel.

In [5]:
import sys
import pandas as pd
import geopandas as gpd
import shapely
import pyproj

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('geopandas:', gpd.__version__)
print('shapely:', shapely.__version__)
print('pyproj:', pyproj.__version__)

Python: 3.11.15
pandas: 3.0.3
geopandas: 1.1.4
shapely: 2.1.2
pyproj: 3.7.2


/home/kanon/miniconda3/envs/calgary-etl/lib/python3.11/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## Step 2: PostGIS Setup (Docker)

**Purpose:** Bring up a reproducible PostGIS database target for the ETL pipeline.

Why this matters:
- PostGIS is your final storage and query engine.
- Docker keeps setup consistent across machines.

Run these in terminal:
```bash
docker compose up -d
docker compose ps
docker compose exec -T postgis psql -U postgres -d postgres -f sql/init.sql
docker compose exec postgis psql -U postgres -d calgary_gis -c "SELECT PostGIS_Version();"
```

The next cell gives you a quick status check from inside Python.

In [7]:
import subprocess

cmd = ['docker', 'compose', 'ps']
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout if result.stdout else result.stderr)

NAME              IMAGE                    COMMAND                  SERVICE   CREATED        STATUS         PORTS
calgary-postgis   postgis/postgis:15-3.4   "docker-entrypoint.s…"   postgis   23 hours ago   Up 3 minutes   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp



## Step 3: Extract Module

**Purpose:** Download raw datasets exactly as published and log provenance metadata.

Why this matters:
- Keeps a stable raw snapshot for reproducibility.
- Log file records source URLs, timestamps, and sizes for auditing.

### Extract Module Reference (Commented)

This cell stores the full `src/extract.py` implementation with comments so you can reuse it later in your portfolio write-up.

How to use this reference:
- Read it top-to-bottom once before running extract.
- Compare it to your own version if you refactor later.
- Keep this as an artifact that demonstrates reproducibility and logging design choices.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import requests

from src.config import DATASETS


# CSV log that records extract provenance for every run.
LOG_PATH = Path("outputs/logs/extract_log.csv")


def ensure_parent_dir(file_path: str) -> None:
    # Create parent folders if they do not exist yet.
    Path(file_path).parent.mkdir(parents=True, exist_ok=True)


def append_log(rows: list[dict]) -> None:
    # Ensure the log directory exists and append rows across runs.
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    file_exists = LOG_PATH.exists()

    with LOG_PATH.open("a", newline="", encoding="utf-8") as f:
        # DictWriter keeps log columns in a stable, explicit order.
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "dataset",
                "source_url",
                "output_path",
                "downloaded_at_utc",
                "http_status",
                "bytes_written",
            ],
        )
        # Write header only once when the log file is first created.
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


def run_extract() -> None:
    # Accumulate one log record per dataset, then write in one operation.
    log_rows = []

    for dataset_name, cfg in DATASETS.items():
        # Read source URL and stable output path from config.
        url = cfg["url"]
        output_path = cfg["output"]  # stable name defined in config.py
        ensure_parent_dir(output_path)

        # Download the raw dataset and fail fast on HTTP errors.
        response = requests.get(url, timeout=60)
        response.raise_for_status()

        # Persist raw bytes exactly as received from the source endpoint.
        data_bytes = response.content
        Path(output_path).write_bytes(data_bytes)

        # Capture extract metadata for reproducibility and troubleshooting.
        log_rows.append(
            {
                "dataset": dataset_name,
                "source_url": url,
                "output_path": output_path,
                "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
                "http_status": response.status_code,
                "bytes_written": len(data_bytes),
            }
        )

        print(f"Saved {dataset_name} -> {output_path} ({len(data_bytes)} bytes)")

    # Write one batch of log rows and print final log location.
    append_log(log_rows)
    print(f"Wrote log -> {LOG_PATH}")


if __name__ == "__main__":
    # Allow direct execution: python -m src.extract
    run_extract()

In [8]:
from src.config import DATASETS

print('Configured datasets:')
for name, cfg in DATASETS.items():
    print(f"- {name}: {cfg['url']} -> {cfg['output']}")

Configured datasets:
- communities: https://data.calgary.ca/api/v3/views/surr-xmvs/query.geojson -> data/raw/communities.geojson
- roads: https://data.calgary.ca/api/v3/views/tqjs-vnhy/query.geojson -> data/raw/roads.geojson
- transit_stops: https://data.calgary.ca/api/v3/views/muzh-c9qc/query.geojson -> data/raw/transit_stops.geojson
- land_use_districts: https://data.calgary.ca/api/v3/views/qe6k-p9nh/query.geojson -> data/raw/land_use_districts.geojson


In [9]:
# Run extract only when you are ready to refresh raw data.
from src.extract import run_extract

run_extract()

Saved communities -> data/raw/communities.geojson (2273705 bytes)
Saved roads -> data/raw/roads.geojson (24758676 bytes)
Saved transit_stops -> data/raw/transit_stops.geojson (4662722 bytes)
Saved land_use_districts -> data/raw/land_use_districts.geojson (22512627 bytes)
Wrote log -> outputs/logs/extract_log.csv


In [10]:
from pathlib import Path

raw_dir = Path('data/raw')
print('Raw files summary:')
for p in sorted(raw_dir.glob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"- {p.name}: {size_kb:.1f} KB")

Raw files summary:
- communities.geojson: 2220.4 KB
- land_use_districts.geojson: 21985.0 KB
- roads.geojson: 24178.4 KB
- transit_stops.geojson: 4553.4 KB


## Step 4: Transform Module

**Purpose:** Convert raw layers into a clean, consistent, analysis-ready spatial model.

What transform should do:
1. Read each raw layer
2. Normalize column names
3. Keep required fields only
4. Repair invalid geometry
5. Reproject to one CRS (recommended: EPSG:3347)
6. Write processed outputs
7. Log row counts and geometry quality stats

Why this matters:
- Prevents CRS mismatches and schema drift.
- Makes QA and PostGIS load reliable and repeatable.

### Transform Module Reference (Commented)

This is a full reference implementation for [src/transform.py](src/transform.py).
Use it as your learning template, then copy into a Python file when ready.

What this implementation handles:
- Schema normalization (lower snake_case)
- Optional field selection per dataset
- Geometry validation and repair
- Consistent reprojection to one CRS
- Process log output to [outputs/logs/transform_log.csv](outputs/logs/transform_log.csv)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import re

import geopandas as gpd

from src.config import DATASETS


# Use one target CRS for all layers so spatial operations are consistent.
TARGET_CRS = "EPSG:3347"

# Optional field subsets by dataset (use normalized names).
# Edit these lists as you learn the exact source schema.
KEEP_FIELDS = {
    "communities": ["name", "class_code", "sector"],
    "roads": ["street_name", "street_type", "quadleft", "quadright"],
    "transit_stops": ["stop_id", "stop_name", "status"],
    "land_use_districts": ["land_use_district", "description"],
}

# If an ID field exists, cast it to string to avoid downstream join/type issues.
ID_FIELDS = {
    "communities": "class_code",
    "roads": None,
    "transit_stops": "stop_id",
    "land_use_districts": "land_use_district",
}

LOG_PATH = Path("outputs/logs/transform_log.csv")


def normalize_col_name(name: str) -> str:
    # Convert to lowercase snake_case for predictable coding/SQL.
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def normalize_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    gdf.columns = [normalize_col_name(c) for c in gdf.columns]
    return gdf


def processed_output_path(raw_output_path: str) -> Path:
    # Convert data/raw/<file> to data/processed/<file>.
    return Path(raw_output_path.replace("data/raw/", "data/processed/"))


def keep_required_fields(
    gdf: gpd.GeoDataFrame, keep_fields: list[str]
 ) -> tuple[gpd.GeoDataFrame, list[str]]:
    gdf = gdf.copy()
    missing = []

    # Ensure expected fields exist; create null fields if missing.
    for field in keep_fields:
        if field not in gdf.columns:
            gdf[field] = None
            missing.append(field)

    return gdf[keep_fields + ["geometry"]], missing


def cast_id_to_string(gdf: gpd.GeoDataFrame, id_field: str | None) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    if id_field and id_field in gdf.columns:
        gdf[id_field] = gdf[id_field].astype("string")
    return gdf


def repair_geom(geom):
    # Try make_valid first; fallback to buffer(0) for compatibility.
    try:
        from shapely import make_valid

        return make_valid(geom)
    except Exception:
        try:
            return geom.buffer(0)
        except Exception:
            return None


def clean_geometry(gdf: gpd.GeoDataFrame) -> tuple[gpd.GeoDataFrame, int, int]:
    gdf = gdf.copy()

    # Remove null and empty geometries before validation.
    gdf = gdf[gdf.geometry.notnull()]
    gdf = gdf[~gdf.geometry.is_empty]

    invalid_before = int((~gdf.is_valid).sum())

    if invalid_before > 0:
        mask = ~gdf.is_valid
        gdf.loc[mask, "geometry"] = gdf.loc[mask, "geometry"].apply(repair_geom)

    # Drop anything still invalid/null/empty after repair attempts.
    gdf = gdf[gdf.geometry.notnull()]
    gdf = gdf[~gdf.geometry.is_empty]
    gdf = gdf[gdf.is_valid]

    invalid_after = int((~gdf.is_valid).sum())
    return gdf, invalid_before, invalid_after


def ensure_crs_and_reproject(gdf: gpd.GeoDataFrame, target_crs: str) -> gpd.GeoDataFrame:
    gdf = gdf.copy()

    # Calgary GeoJSON is typically EPSG:4326. Set only when missing.
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    return gdf.to_crs(target_crs)


def append_transform_log(rows: list[dict]) -> None:
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    file_exists = LOG_PATH.exists()

    with LOG_PATH.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "dataset",
                "input_path",
                "output_path",
                "rows_in",
                "rows_out",
                "missing_fields",
                "invalid_before",
                "invalid_after",
                "crs_out",
                "processed_at_utc",
                "status",
                "error",
            ],
        )
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


def process_dataset(dataset_name: str, cfg: dict) -> dict:
    input_path = cfg["output"]
    output_path = processed_output_path(input_path)

    row = {
        "dataset": dataset_name,
        "input_path": input_path,
        "output_path": str(output_path),
        "rows_in": 0,
        "rows_out": 0,
        "missing_fields": "",
        "invalid_before": 0,
        "invalid_after": 0,
        "crs_out": TARGET_CRS,
        "processed_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "ok",
        "error": "",
    }

    try:
        if not Path(input_path).exists():
            raise FileNotFoundError(f"Missing raw file: {input_path}")

        gdf = gpd.read_file(input_path)
        row["rows_in"] = len(gdf)

        gdf = normalize_columns(gdf)

        keep_fields = KEEP_FIELDS.get(dataset_name, [])
        keep_fields = [f for f in keep_fields if f != "geometry"]
        if keep_fields:
            gdf, missing = keep_required_fields(gdf, keep_fields)
            row["missing_fields"] = ",".join(missing)

        id_field = ID_FIELDS.get(dataset_name)
        gdf = cast_id_to_string(gdf, id_field)

        gdf, invalid_before, invalid_after = clean_geometry(gdf)
        row["invalid_before"] = invalid_before
        row["invalid_after"] = invalid_after

        gdf = ensure_crs_and_reproject(gdf, TARGET_CRS)

        output_path.parent.mkdir(parents=True, exist_ok=True)
        gdf.to_file(output_path, driver="GeoJSON")
        row["rows_out"] = len(gdf)

        print(
            f"{dataset_name}: rows_in={row['rows_in']} rows_out={row['rows_out']} "
            f"invalid_before={invalid_before} invalid_after={invalid_after} "
            f"crs={TARGET_CRS} -> {output_path}"
        )

    except Exception as exc:
        row["status"] = "error"
        row["error"] = str(exc)
        print(f"{dataset_name}: ERROR -> {exc}")

    return row


def run_transform() -> None:
    # Run the same transform pipeline for each configured dataset.
    log_rows = []
    for dataset_name, cfg in DATASETS.items():
        log_rows.append(process_dataset(dataset_name, cfg))

    append_transform_log(log_rows)
    print(f"Wrote log -> {LOG_PATH}")


if __name__ == "__main__":
    # Allow direct execution: python -m src.transform
    run_transform()

In [ ]:
# Optional: run this after you create src/transform.py
from pathlib import Path

transform_path = Path('src/transform.py')
if transform_path.exists():
    from src.transform import run_transform
    run_transform()
else:
    print('src/transform.py not found yet. Create it first, then rerun this cell.')

In [ ]:
# Quick check of processed outputs
from pathlib import Path
import geopandas as gpd

processed_dir = Path('data/processed')
files = sorted(processed_dir.glob('*.geojson'))

if not files:
    print('No processed files found yet.')
else:
    for p in files:
        gdf = gpd.read_file(p)
        invalid = int((~gdf.is_valid).sum()) if len(gdf) else 0
        print(f"{p.name}: rows={len(gdf)}, crs={gdf.crs}, invalid_geom={invalid}")

## Learning Checkpoint

By this point, you should understand:
- Why raw data is preserved before transformation
- How schema standardization improves reliability
- Why geometry validity and CRS consistency are critical
- How logs support reproducibility and QA evidence

If any result looks unexpected, pause and inspect one layer end-to-end before continuing.

## Step 5: QA Module

**Purpose:** Validate processed data quality and publish repeatable QA evidence.

What QA should check:
1. Row count by layer
2. Invalid geometry count
3. Null count in key fields
4. Duplicate IDs where an ID field exists
5. CRS consistency

Why this matters:
- QA protects downstream analysis from silent data issues.
- A QA report makes your ETL process auditable and portfolio-ready.

In [ ]:
# Run the blocking QA gate against all processed layers.
from src.qa import QualityGateError, run_qa

try:
    qa_results = run_qa()
except QualityGateError as error:
    print(f"LOAD BLOCKED: {error}")
    raise
else:
    print(f"QA approved {len(qa_results)} layers for loading.")

In [ ]:
# Inspect QA outputs
from pathlib import Path
import pandas as pd

qa_csv = Path('outputs/qa/qa_report.csv')
if qa_csv.exists():
    df = pd.read_csv(qa_csv)
    print(df.head())
    print('')
    print(f'Rows in QA report: {len(df)}')
else:
    print('No QA report found at outputs/qa/qa_report.csv yet.')

### QA Learning Notes

Interpret results like this:
- **High invalid geometry count:** adjust transform geometry repair logic.
- **High nulls in key fields:** source data may be sparse, or field mapping needs updates.
- **Duplicate IDs:** review ID standardization and uniqueness assumptions.
- **Mixed CRS values:** ensure every output layer is reprojected in transform.

## Step 6: Load Module (PostGIS)

**Purpose:** Load cleaned spatial layers into PostGIS and create spatial indexes.

What load should do:
1. Connect to the database
2. Write each processed layer to PostGIS
3. Create GIST spatial indexes on geometry
4. Run ANALYZE to improve query planning

Why this matters:
- PostGIS turns files into queryable production-style spatial tables.
- Indexes are essential for fast spatial operations and map services.

In [ ]:
# Load QA-approved layers transactionally and verify each result.
# Requires the PostGIS service and initialized calgary_gis database.
from src.load import run_load

load_results = run_load()
for result in load_results:
    print(
        f"{result.table_name}: rows={result.loaded_rows}, "
        f"srid={result.srid}, spatial_index={result.spatial_index_present}"
    )

In [ ]:
# Quick database sanity checks via docker compose
import subprocess

commands = [
    ['docker', 'compose', 'exec', '-T', 'postgis', 'psql', '-U', 'postgres', '-d', 'calgary_gis', '-c', '\dt'],
    ['docker', 'compose', 'exec', '-T', 'postgis', 'psql', '-U', 'postgres', '-d', 'calgary_gis', '-c', 'SELECT PostGIS_Version();'],
]

for cmd in commands:
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(' '.join(cmd))
    print(result.stdout if result.stdout else result.stderr)
    print('-' * 80)

### PostGIS Learning Notes

Check these success signals:
- Your tables appear in `\dt` output.
- Geometry columns exist and are typed correctly.
- Spatial indexes exist on geometry columns.
- Queries run quickly compared to non-indexed layers.

If load fails, common causes are:
- Docker service not running
- Wrong database connection string
- Invalid geometry not fully resolved in transform

## Step 7: Run and Verify the Complete Pipeline

With Docker running and `calgary_gis` initialized, execute the complete pipeline from the project root:

```bash
python -m src.main
```

To reuse existing raw snapshots while rerunning Transform, QA, and Load:

```bash
python -m src.main --skip-extract
```

Run deterministic tests first, then include isolated PostGIS integration tests:

```bash
python -m unittest discover -s tests -v
RUN_POSTGIS_TESTS=1 python -m unittest discover -s tests -v
```

Release-ready evidence includes passing QA results, matching source and loaded row counts, SRID `3347`, GIST geometry indexes, repeatable replacement without duplicates, and rollback after a forced failure.